# Explore Sentiment of Corpus using a lexicon

Notes:

I am following along with my M10 Sentiment Analysis of Novels Notes as I complete this section.

## Setup

### Import Libraries

In [49]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

### Configure

In [50]:
# specify OHCO and bags
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

bags = dict(
    SENTS = OHCO[:4],
    PARAS = OHCO[:3],
    CHAPS = OHCO[:2],
    BOOKS = OHCO[:1]
)

In [51]:
# set chapter as bag
bag = "CHAPS"

In [52]:
# # Subset of columns from salex
# emo_cols = "anger anticipation disgust fear joy sadness surprise trust sentiment".split()

### Load Data

In [53]:
# get NRC lexicon
SALEX = pd.read_csv('data/salex_nrc.csv').set_index('term_str')
SALEX.columns = [col.replace('nrc_','') for col in SALEX.columns]

SALEX

,anger,anticipation,disgust,fear,joy,negative,positive,sadness,surprise,trust,sentiment
term_str,,,,,,,,,,,
abandon,0,0,0,1,0,1,0,1,0,0,-1
abandoned,1,0,0,1,0,1,0,1,0,0,-1
abandonment,1,0,0,1,0,1,0,1,1,0,-1
abduction,0,0,0,1,0,1,0,1,1,0,-1
aberration,0,0,1,0,0,1,0,0,0,0,-1
...,...,...,...,...,...,...,...,...,...,...,...
young,0,1,0,0,1,0,1,0,1,0,1
youth,1,1,0,1,1,0,1,0,1,0,1
zeal,0,1,0,0,1,0,1,0,1,1,1


This is a sentiment analysis table.

(Future attempts may use SentiWordNet instead.)

In [54]:
# load in tables
LIB = pd.read_csv('data/LIB.csv', sep='\t').set_index('book_id')
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)
VOCAB = pd.read_csv('data/VOCAB.csv', sep='\t').set_index('term_str')

## Apply the Model

In [55]:
# join sentiment analysis table (SALEX) to VOCAB table
# filter out stop words and filter for POS that carry sentiment
sentiment_pos_groups = ['JJ', 'NN', 'VB', 'RB']

VOCAB_SENT = VOCAB[
    (VOCAB['stop'] == False) & 
    (VOCAB['max_pos_group'].isin(sentiment_pos_groups))
].join(SALEX, on='lemma', how='inner')

VOCAB_SENT

,n,n_chars,p,i,max_pos,max_pos_group,lemma,n_pos_group,n_pos,stop,...,anticipation,disgust,fear,joy,negative,positive,sadness,surprise,trust,sentiment
term_str,,,,,,,,,,,,,,,,,,,,,
abandon,6,7,0.000007,17.104609,VB,VB,abandon,2,3,False,...,0,0,1,0,1,0,1,0,0,-1
abandoned,19,9,0.000022,15.441644,VBN,VB,abandon,1,2,False,...,0,0,1,0,1,0,1,0,0,-1
abandoning,7,10,0.000008,16.882217,VBG,VB,abandon,1,1,False,...,0,0,1,0,1,0,1,0,0,-1
abandonment,1,11,0.000001,19.689571,NN,NN,abandonment,1,1,False,...,0,0,1,0,1,0,1,1,0,-1
abduction,4,9,0.000005,17.689571,NN,NN,abduction,1,1,False,...,0,0,1,0,1,0,1,1,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
youth,34,5,0.000040,14.602109,NN,NN,youth,2,3,False,...,1,0,1,1,0,1,0,1,0,1
youths,1,6,0.000001,19.689571,NNS,NN,youth,1,1,False,...,1,0,1,1,0,1,0,1,0,1
zeal,7,4,0.000008,16.882217,NN,NN,zeal,1,1,False,...,1,0,0,1,0,1,0,1,1,1


In [56]:
# what percent of corpus tokens is my sentiment analysis seeing
matched_tokens = VOCAB.loc[VOCAB_SENT.index, 'n'].sum()
total_tokens = VOCAB['n'].sum()
print(matched_tokens / total_tokens)

0.06668606177578781


In [57]:
content_vocab = VOCAB[VOCAB['stop'] == False]
matched_content = content_vocab.loc[content_vocab.index.intersection(VOCAB_SENT.index), 'n'].sum()
total_content = content_vocab['n'].sum()
print(matched_content / total_content)

0.14188837249094766


Original run checks:
- 0.055264933891233034 of corpus tokens my SA is seeing
- 0.12115644893191092 of content word tokens are covered

It appears that I will need to lemmatize to get better coverage of content word tokens. I shall be adding spaCy lemmatization to my CORPUS notebook. I will in fact be taking this as my motivation to swap to spaCy en_core_web_lg.

I made small improvements to 6.7% overall coverage and 14.2% coverage on content words.

For transparency, it is worth noting that NRC lexicon covers approximately 14% of content word tokens in my corpus, reflecting both the lexicon's general-English scope and spaCy's POS-driven lemmatization, which ends up classifying many emotionally relevant adjectives as non-verbal forms which don't reduce to their base entries in NRC.

In [58]:
VOCAB_SENT.sort_values('n', ascending=False).head(20)[['n', 'lemma', 'max_pos', 'sentiment']]

,n,lemma,max_pos,sentiment
term_str,,,,
sir,1228,sir,NNP,1
good,1210,good,JJ,1
young,745,young,JJ,1
left,706,leave,VBD,-1
kind,593,kind,NN,1
friend,581,friend,NN,1
case,573,case,NN,-1
battle,443,battle,NNP,-1
lord,441,lord,NNP,0


In [59]:
CORPUS[CORPUS.term_str == 'battle'][['token_str', 'pos', 'lemma']].head(10)

token_str  \
book_id                     chap_num para_num sent_num token_num             
giants-bread                5        84       2        7            battle   
the-murder-of-roger-ackroyd 17       145      33       5            battle   
the-secret-adversary        23       24       2        3            battle   
the-secret-of-chimneys      10       131      3        3            Battle   
                                     132      0        0            Battle   
                                     133      1        4            Battle   
                            11       13       0        1            Battle   
                                     14       0        16           Battle   
                                     15       0        5            Battle   
                                     21       0        0            Battle   

                                                                  pos   lemma  
book_id                     chap_num para_num sent_num token_num               
giants-bread                5        84       2        7           NN  battle  
the-murder-of-roger-ackroyd 17       145      33       5           NN  battle  
the-secret-adversary        23       24       2        3           NN  battle  
the-secret-of-chimneys      10       131      3        3          NNP  battle  
                                     132      0        0           NN  battle  
                                     133      1        4          NNP  battle  
                            11       13       0        1          NNP  battle  
                                     14       0        16         NNP  battle  
                                     15       0        5          NNP  battle  
                                     21       0        0          NNP  battle

'battle' primarily is a reference to Superintendent Battle

In [60]:
CORPUS[CORPUS.term_str == 'sir'][['token_str', 'pos']].head(5)

token_str  pos
book_id      chap_num para_num sent_num token_num               
giants-bread 7        85       1        4               Sir  NNP
                      86       8        3               Sir  NNP
             8        14       4        3               Sir  NNP
             24       23       0        1               sir   NN
                      25       0        7               sir   NN

'sir' scores positive 1228 times and is a form of address.

I will filter POS to only sentiment-bearing categories in order to address these limitations.

In [62]:
# how many terms survived?
print(len(VOCAB_SENT))

# confirm proper nouns are gone
print(VOCAB_SENT[VOCAB_SENT['max_pos'] == 'NNP'])  # should be empty

# rerun coverage
matched = VOCAB.loc[VOCAB[VOCAB.stop == False].index.intersection(VOCAB_SENT.index), 'n'].sum()
total_content = VOCAB[VOCAB.stop == False]['n'].sum()
print(matched / total_content)

3517
              n  n_chars         p          i max_pos max_pos_group  \
term_str                                                              
allied        3        6  0.000004  18.104609     NNP            NN   
antichrist    1       10  0.000001  19.689571     NNP            NN   
aunt        101        4  0.000119  13.031360     NNP            NN   
baited        2        6  0.000002  18.689571     NNP            VB   
battle      443        6  0.000524  10.898409     NNP            NN   
...         ...      ...       ...        ...     ...           ...   
united       10        6  0.000012  16.367643     NNP            NN   
university    1       10  0.000001  19.689571     NNP            NN   
victor       62        6  0.000073  13.735375     NNP            NN   
wan           3        3  0.000004  18.104609     NNP            NN   
wot           2        3  0.000002  18.689571     NNP            NN   

                 lemma  n_pos_group  n_pos   stop  ... anticipation  di

## Save Outputs

In [61]:
# # save the DCM table to csv
# DCM.to_csv('data/PCA_DCM.csv', sep='\t', index=True)

# # save the LOADINGS table to csv
# LOADINGS.to_csv('data/PCA_LOADINGS.csv', sep='\t', index=True)

# # save the DOC table to csv
# DOC.to_csv('data/PCA_DOC.csv', sep='\t', index=True)